# Estudo de Não Recorrência — Consignado | Bradesco Expresso

## Objetivo
Este notebook cria a base inicial para investigar lojas que **produziram Consignado até o mesmo dia útil do mês anterior e ainda não produziram no mês atual**.

A análise foi desenhada para evoluir de **identificação da coorte → comparação entre grupos → comportamento em outros produtos → histórico → território/hierarquia → hipóteses de perda → priorização comercial**.

### Princípio metodológico
Não analisar somente as lojas perdidas. O grupo que **manteve produção** será usado como comparação para entender quais características diferenciam os dois comportamentos.

> Ajuste os nomes das colunas de hierarquia/território conforme sua tabela real.


In [1]:
# 1. Bibliotecas e parâmetros
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from db import read_sql

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# =========================
# PARÂMETROS DO ESTUDO
# =========================
PERIODO_ANTERIOR = 202607
PERIODO_ATUAL    = 202608
DU_CORTE         = 14

# Produtos disponíveis
PRODUTOS = {
    "CONTABIL": ("QTD_TRX", "QTD_TRX_ACUM"),
    "CRED_TOTAL": ("CRED_TOTAL", "CRED_TOTAL_ACUM"),
    "CREDITO_JORNADA": ("CREDITO_JORNADA", "CREDITO_JORNADA_ACUM"),
    "PARCELADO": ("VLR_CREDITO_PARCEL", "VLR_CREDITO_PARCEL_ACUM"),
    "CONSIG": ("VLR_CONSIG", "VLR_CONSIG_ACUM"),
    "LIME": ("VLR_LIME", "VLR_LIME_ACUM"),
    "CONTAS": ("CONTAS", "CONTAS_ACUM"),
    "CARTAO_CREDITO": ("QTD_CARTAO_CONTRATADO", "QTD_CARTAO_CONTRATADO_ACUM"),
    "SEGUROS": ("SEG_TOTAL", "SEG_TOTAL_ACUM"),
}

print(f"Comparação: {PERIODO_ANTERIOR} x {PERIODO_ATUAL} | até {DU_CORTE}º DU")


Comparação: 202607 x 202608 | até 13º DU


## 2. Extração da `TESTE..PRODUCAO_DIA_UTIL_BE`

A consulta abaixo traz somente os períodos necessários para a primeira comparação.  
A conexão SQL fica em `db.py` — ajuste `SERVER` / `DATABASE` lá uma vez.


In [ ]:
# 2. Extração (conexão em db.py)

query = f"""
SELECT
    *
FROM TESTE..PRODUCAO_DIA_UTIL_BE
WHERE PERIODO IN ({PERIODO_ANTERIOR}, {PERIODO_ATUAL})
  AND QTD_DIA_UTIL_MES <= {DU_CORTE}
"""

print(query)

df = read_sql(query)

# Alternativa local (sem SQL):
# df = pd.read_csv("PRODUCAO_DIA_UTIL_BE.csv")


## 3. Auditoria da base

Antes de qualquer insight, validamos granularidade, duplicidades, dias úteis e consistência dos acumulados.  
A expectativa é existir no máximo uma linha por `CHAVE_LOJA + PERIODO + QTD_DIA_UTIL_MES`.


In [ ]:
def auditar_base(df):
    print("Linhas:", len(df))
    print("Lojas:", df["CHAVE_LOJA"].nunique())
    print("Períodos:", sorted(df["PERIODO"].dropna().unique()))

    dup = df.duplicated(
        ["CHAVE_LOJA", "PERIODO", "QTD_DIA_UTIL_MES"],
        keep=False
    )
    print("Linhas duplicadas na granularidade esperada:", dup.sum())

    resumo = (
        df.groupby("PERIODO")
          .agg(
              lojas=("CHAVE_LOJA", "nunique"),
              du_min=("QTD_DIA_UTIL_MES", "min"),
              du_max=("QTD_DIA_UTIL_MES", "max")
          )
    )
    display(resumo)

# auditar_base(df)


## 4. Snapshot comparável no mesmo dia útil

Para classificar corretamente as lojas, usamos o **acumulado até o 13º dia útil** nos dois períodos.

Isso evita comparar agosto parcial com julho fechado.


In [ ]:
def snapshot_du(df, periodo, du=14):
    base = df[
        (df["PERIODO"] == periodo) &
        (df["QTD_DIA_UTIL_MES"] == du)
    ].copy()

    # Caso existam duplicidades, esta validação impede análise silenciosamente errada.
    if base.duplicated("CHAVE_LOJA").any():
        raise ValueError(
            f"Existem CHAVE_LOJA duplicadas em {periodo} no {du}º DU."
        )

    return base

# jul14 = snapshot_du(df, PERIODO_ANTERIOR, DU_CORTE)
# ago14 = snapshot_du(df, PERIODO_ATUAL, DU_CORTE)


## 5. Construção da coorte

A população será dividida em quatro estados:

| Julho até 13º DU | Agosto até 13º DU | Estado |
|---|---|---|
| Produziu | Produziu | **MANTEVE** |
| Produziu | Não produziu | **PERDEU** |
| Não produziu | Produziu | **ENTROU** |
| Não produziu | Não produziu | INATIVA |

Para investigar as **502 perdas**, o grupo mais importante de comparação é `MANTEVE`.


In [ ]:
def montar_coorte(jul, ago):
    cols_jul = ["CHAVE_LOJA"] + [
        acum for _, acum in PRODUTOS.values()
        if acum in jul.columns
    ]
    cols_ago = ["CHAVE_LOJA"] + [
        acum for _, acum in PRODUTOS.values()
        if acum in ago.columns
    ]

    j = jul[cols_jul].copy()
    a = ago[cols_ago].copy()

    j = j.rename(columns={
        c: f"{c}_ANT" for c in j.columns if c != "CHAVE_LOJA"
    })
    a = a.rename(columns={
        c: f"{c}_ATUAL" for c in a.columns if c != "CHAVE_LOJA"
    })

    base = j.merge(a, on="CHAVE_LOJA", how="outer").fillna(0)

    base["PROD_CONSIG_ANT"] = (
        base.get("QTD_CONSIG_ACUM_ANT", 0) > 0
    ).astype(int)

    base["PROD_CONSIG_ATUAL"] = (
        base.get("QTD_CONSIG_ACUM_ATUAL", 0) > 0
    ).astype(int)

    condicoes = [
        (base["PROD_CONSIG_ANT"] == 1) & (base["PROD_CONSIG_ATUAL"] == 1),
        (base["PROD_CONSIG_ANT"] == 1) & (base["PROD_CONSIG_ATUAL"] == 0),
        (base["PROD_CONSIG_ANT"] == 0) & (base["PROD_CONSIG_ATUAL"] == 1),
    ]
    escolhas = ["MANTEVE", "PERDEU", "ENTROU"]

    base["STATUS_CONSIG"] = np.select(
        condicoes, escolhas, default="INATIVA"
    )

    return base

# coorte = montar_coorte(jul14, ago14)
# coorte["STATUS_CONSIG"].value_counts()


## 6. Validação das 502

Este é um checkpoint importante. Antes de avançar, a quantidade classificada como `PERDEU` deve coincidir com o número esperado da análise.


In [ ]:
# perdas = coorte.query("STATUS_CONSIG == 'PERDEU'").copy()
# mantidas = coorte.query("STATUS_CONSIG == 'MANTEVE'").copy()

# print("Lojas perdidas:", len(perdas))
# print("Lojas mantidas:", len(mantidas))

# ESPERADO_PERDAS = 502
# if len(perdas) != ESPERADO_PERDAS:
#     print(
#         f"ATENÇÃO: esperado={ESPERADO_PERDAS}, encontrado={len(perdas)}. "
#         "Revise granularidade, conceito e disponibilidade do 13º DU."
#     )


## 7. Comportamento das perdas nos outros produtos

Aqui começa a primeira pergunta analítica:

> **A loja deixou somente Consignado ou o relacionamento comercial inteiro está desacelerando?**

Para cada produto, calculamos valor anterior, atual, variação absoluta e percentual no mesmo estágio do mês.


In [ ]:
def adicionar_variacoes(base):
    out = base.copy()

    for produto, (_, acum) in PRODUTOS.items():
        ant = f"{acum}_ANT"
        atual = f"{acum}_ATUAL"

        if ant not in out.columns or atual not in out.columns:
            continue

        out[f"{produto}_DELTA"] = out[atual] - out[ant]

        # NaN quando base anterior = 0 para evitar percentuais artificiais.
        out[f"{produto}_VAR_PCT"] = np.where(
            out[ant] != 0,
            (out[atual] / out[ant] - 1) * 100,
            np.nan
        )

    return out

# coorte = adicionar_variacoes(coorte)
# perdas = coorte.query("STATUS_CONSIG == 'PERDEU'").copy()


In [ ]:
def classificar_movimento(atual, anterior, tolerancia=0.10):
    """Classificação simples; ajuste a tolerância por produto se necessário."""
    if pd.isna(atual) or pd.isna(anterior):
        return "SEM DADO"
    if anterior == 0:
        return "ENTROU" if atual > 0 else "SEM PRODUÇÃO"

    variacao = atual / anterior - 1

    if variacao > tolerancia:
        return "CRESCENDO"
    if variacao < -tolerancia:
        return "CAINDO"
    return "ESTÁVEL"


def classificar_outros_produtos(base):
    out = base.copy()

    for produto, (_, acum) in PRODUTOS.items():
        if produto == "CONSIG":
            continue

        ant = f"{acum}_ANT"
        atual = f"{acum}_ATUAL"

        if ant in out.columns and atual in out.columns:
            out[f"STATUS_{produto}"] = [
                classificar_movimento(a, b)
                for a, b in zip(out[atual], out[ant])
            ]

    return out

# perdas = classificar_outros_produtos(perdas)


## 8. Matriz executiva de comportamento

Em vez de olhar somente médias, medimos **qual percentual das lojas perdidas está crescendo, estável ou caindo em cada outro produto**.


In [ ]:
def matriz_comportamento(perdas):
    colunas = [
        c for c in perdas.columns
        if c.startswith("STATUS_") and c != "STATUS_CONSIG"
    ]

    resultado = []

    for col in colunas:
        produto = col.replace("STATUS_", "")
        dist = perdas[col].value_counts(normalize=True).mul(100)

        linha = {"PRODUTO": produto}
        linha.update(dist.to_dict())
        resultado.append(linha)

    return pd.DataFrame(resultado).fillna(0)

# matriz = matriz_comportamento(perdas)
# display(matriz)


## 9. Visual inicial — o problema é específico de Consignado?

Este gráfico ajuda a identificar se as 502 lojas continuam comercialmente ativas em outros produtos.


In [ ]:
def plot_comportamento(matriz):
    categorias = ["CRESCENDO", "ESTÁVEL", "CAINDO"]
    existentes = [c for c in categorias if c in matriz.columns]

    if not existentes:
        print("Sem classificações suficientes para o gráfico.")
        return

    plot_df = matriz.set_index("PRODUTO")[existentes]

    ax = plot_df.plot(
        kind="barh",
        stacked=True,
        figsize=(11, 5)
    )

    ax.set_title(
        "Comportamento das lojas sem recorrência de Consignado",
        loc="left",
        fontweight="bold"
    )
    ax.set_xlabel("% das lojas")
    ax.set_ylabel("")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.show()

# plot_comportamento(matriz)


## 10. Histórico de recorrência — próxima evolução

A comparação Jul × Ago não é suficiente. Uma loja que produziu Consignado em apenas um dos últimos seis meses é diferente de uma loja que produziu em todos os seis e interrompeu agora.

Para a próxima etapa, extraia pelo menos **6 meses anteriores** e calcule:

- meses com Consignado nos últimos 6 meses;
- dias com produção;
- produção média e mediana;
- ticket histórico;
- tendência;
- quantidade de produtos ativos;
- estabilidade transacional.

A célula abaixo já deixa uma função inicial para recorrência mensal.


In [ ]:
def features_historicas(df_hist):
    mensal = (
        df_hist.groupby(["CHAVE_LOJA", "PERIODO"], as_index=False)
        .agg(
            CONSIG=("QTD_CONSIG", "sum"),
            CONTAS=("QTD_CONTAS", "sum"),
            SEGUROS=("SEG_TOTAL", "sum"),
            CRED_TOTAL=("CRED_TOTAL", "sum"),
            LIME=("LIME", "sum"),
        )
    )

    mensal["PROD_CONSIG"] = (mensal["CONSIG"] > 0).astype(int)

    features = (
        mensal.groupby("CHAVE_LOJA")
        .agg(
            CONSIG_MESES_PROD=("PROD_CONSIG", "sum"),
            CONSIG_MEDIA=("CONSIG", "mean"),
            CONSIG_MEDIANA=("CONSIG", "median"),
            CRED_TOTAL_MEDIA=("CRED_TOTAL", "mean"),
            LIME_MEDIA=("LIME", "mean"),
            CONTAS_MEDIA=("CONTAS", "mean"),
            SEGUROS_MEDIA=("SEGUROS", "mean"),
        )
        .reset_index()
    )

    return features

# Exemplo futuro:
# df_hist = read_sql("""
# SELECT *
# FROM TESTE..PRODUCAO_DIA_UTIL_BE
# WHERE PERIODO BETWEEN 202602 AND 202607
# """)
#
# historico = features_historicas(df_hist)
# perdas = perdas.merge(historico, on="CHAVE_LOJA", how="left")


## 11. Território e hierarquia comercial

Não devemos assumir que a dinâmica seja nacionalmente homogênea.

Quando as colunas estiverem disponíveis, incorporar:

- `UF`
- `MUNICIPIO`
- população / porte do município
- capital × interior
- Gerência de Gestão
- Gerente Comercial III
- Gerente Comercial

O indicador mais importante não será somente **quantidade de perdas**, mas a **taxa de não recorrência**:

`lojas que perderam / lojas que haviam produzido no período anterior`

Assim estruturas de tamanhos diferentes tornam-se comparáveis.


In [ ]:
def taxa_perda_por_dimensao(base, dimensao):
    elegiveis = base[base["PROD_CONSIG_ANT"] == 1].copy()
    elegiveis["FLAG_PERDA"] = (elegiveis["STATUS_CONSIG"] == "PERDEU").astype(int)

    resumo = (
        elegiveis.groupby(dimensao, dropna=False)
        .agg(
            BASE_ANTERIOR=("CHAVE_LOJA", "nunique"),
            LOJAS_PERDIDAS=("FLAG_PERDA", "sum")
        )
        .reset_index()
    )

    resumo["TX_NAO_RECORRENCIA"] = (
        resumo["LOJAS_PERDIDAS"] /
        resumo["BASE_ANTERIOR"] * 100
    )

    return resumo.sort_values("TX_NAO_RECORRENCIA", ascending=False)

# Após incorporar dimensão territorial:
# display(taxa_perda_por_dimensao(coorte, "UF"))
# display(taxa_perda_por_dimensao(coorte, "GER_GESTAO"))
# display(taxa_perda_por_dimensao(coorte, "GC_III"))


## 12. Hipóteses comerciais iniciais

Não tratar estas classificações como causa comprovada. Elas são **hipóteses comportamentais** para direcionar investigação.

Exemplos:

1. **Perda específica de Consignado** — demais produtos preservados.
2. **Mudança de mix** — Consignado desaparece e LIME/outro crédito cresce.
3. **Deterioração ampla** — vários produtos e movimentação caem.
4. **Produção eventual** — baixa recorrência histórica de Consignado.
5. **Ruptura de recorrente** — forte histórico e interrupção abrupta.
6. **Possível efeito territorial** — taxa de perda elevada também entre pares da praça.


In [ ]:
def hipotese_inicial(row):
    # Regras iniciais deliberadamente simples.
    # Devem ser refinadas após conhecer distribuições reais.

    outros = [
        row.get("STATUS_CONTAS"),
        row.get("STATUS_SEGUROS"),
        row.get("STATUS_CRED_TOTAL"),
        row.get("STATUS_LIME"),
    ]

    outros_validos = [x for x in outros if pd.notna(x)]

    if not outros_validos:
        return "SEM DADOS SUFICIENTES"

    crescendo_estavel = sum(
        x in ["CRESCENDO", "ESTÁVEL", "ENTROU"]
        for x in outros_validos
    )
    caindo = sum(x == "CAINDO" for x in outros_validos)

    if row.get("STATUS_LIME") == "CRESCENDO" and row.get("STATUS_CRED_TOTAL") in ["CRESCENDO", "ESTÁVEL"]:
        return "POSSÍVEL MUDANÇA DE MIX"

    if caindo >= 3:
        return "DETERIORAÇÃO AMPLA"

    if crescendo_estavel >= 3:
        return "PERDA ESPECÍFICA DE CONSIGNADO"

    return "COMPORTAMENTO MISTO"

# perdas["HIPOTESE_INICIAL"] = perdas.apply(hipotese_inicial, axis=1)
# perdas["HIPOTESE_INICIAL"].value_counts()


## 13. Próximos passos do estudo

A evolução recomendada deste notebook é:

**Fase 1 — Coorte**
- validar as 502;
- identificar mantidas e entradas;
- comparar mesmo dia útil.

**Fase 2 — Comportamento**
- analisar outros produtos;
- identificar mudança de mix;
- medir deterioração ou manutenção do relacionamento.

**Fase 3 — Histórico**
- 6 a 12 meses;
- recorrência;
- potencial histórico;
- produção eventual × ruptura.

**Fase 4 — Território**
- comparar UF, município e porte;
- Gerência de Gestão, GC III e GC;
- construir benchmarks entre lojas comparáveis.

**Fase 5 — Explicação**
- comparar `PERDEU × MANTEVE`;
- regressão logística / árvores;
- avaliar drivers diferentes por território.

**Fase 6 — Ação**
- score de oportunidade;
- priorização das lojas;
- direcionamento para a estrutura comercial.


In [ ]:
# 14. Exportações sugeridas

# Path("outputs").mkdir(exist_ok=True)

# perdas.to_excel(
#     "outputs/lojas_nao_recorrentes_consignado.xlsx",
#     index=False
# )

# matriz.to_excel(
#     "outputs/matriz_comportamento_produtos.xlsx",
#     index=False
# )

print("Notebook estruturado. Configure a conexão e execute as etapas em sequência.")
